# flowDEM

<h3> First, let's set up the input parameters</h3>

In [4]:
import subprocess
import utm
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import h5py

In [5]:
modeldx = 30

demdx = 30

safetyMargin = 4

bufferSize = 20

file_name = 'hawaii.mat'

saveStep = 360

ventLat, ventLon = utm.to_latlon(234392, 2159562, 5, 'Q')

fissureLength = 250

ventAzimuth = 80

viscosity = 1e4

effusionRate = 120

eruptionDuration = 86400

In [6]:
f = open("flowDEM_input.py","r+")
conf = f"""from flowDEM import flowDEM

# Resolution of the numerical model
modelDeltax = {modeldx}
# Resolution of the DEM
DEMDeltax = {demdx}

safetyMargin = {safetyMargin}
bufferSize = {bufferSize}

filename = '{file_name}'
saveStep = {saveStep}          # Save the solution every saveStep seconds
saveStepRestart = 3600 * 4

latVent = {ventLat}
lonVent = {ventLon}

LVent = {fissureLength}     # Length of the fissure in meters. If LVent = 0 only one vent will be considered
aziVent = {ventAzimuth}
L = 0           # Smoothing parameter in meters

# Eruption parameters
days = {eruptionDuration/86400}     # End eruption
mu = {viscosity}    # Viscosity
q = {effusionRate}         # Effusion rate

flowDEM(
    filename, saveStep, saveStepRestart,
    lonVent, latVent, LVent, aziVent,
    modelDeltax, DEMDeltax, L,
    days, mu, q,
    safetyMargin, bufferSize
)
"""
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

In [1]:
!python geotiff_to_mat.py

Reading MaunaLoa_30m_UTM5N.asc ...


  Grid size : 1627 rows × 2350 cols
  CRS       : EPSG:32605


  NoData    : -999999.0 → NaN  (0 cells)
  Reprojecting to WGS-84 ...


  Origin    : lat0=19.370610°, lon0=-155.680911°
  x range   : 0.0 – 70580.2 m
  y range   : 0.0 – 49019.3 m


Saved → big.mat  (91.8 MB)


In [7]:
!python flowDEM_input.py

Running flowDEM in directory
/home/jovyan/flowDEM
Not smoothing!
Figure(640x480)

✓ DEM4julia.mat saved successfully
  Grid size: 1773 x 1657
  Vent location: (-3943.2, -10060.9) meters
  Fissure length: 250 m at azimuth 80°
  Simulation duration: 1.0 days (86400.0 seconds)
  Viscosity: 10000.0 Pa·s
  Effusion rate: 120 m³/s


In [9]:
import os
import subprocess

env = os.environ.copy()
env["JULIA_DEPOT_PATH"] = "/home/jovyan/shared/Libraries/julia-1.11.4/depot"
env["JULIA_PROJECT"] = "/home/jovyan/shared/Libraries/julia-1.11.4/environments/v1.11"

subprocess.run(
    ["julia", "--threads=4", "flowDEM.jl"],
    env=env,
    check=True
)

CalledProcessError: Command '['julia', '--threads=4', 'flowDEM.jl']' died with <Signals.SIGKILL: 9>.

In [15]:
# DEM
dem = loadmat('/home/jovyan/flowDEM/DEM4julia.mat')

x = dem['x'].squeeze()
y = dem['y'].squeeze()
DEMup = dem['DEMup']
xs = dem['xs'].squeeze()
ys = dem['ys'].squeeze()

# Results
with h5py.File('/home/jovyan/flowDEM/results.mat', 'r') as f:
    t = np.ravel(np.array(f['t']))
    hsol = np.array(f['hsol'])

# Convert from (nx, ny, nt) -> (nt, ny, nx)
hsol = np.transpose(hsol, (2, 1, 0))

thresh = 0.01
hsol = hsol.astype(float)
hsol[hsol <= thresh] = np.nan

hmax = np.nanmax(hsol)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for i, ax in enumerate(axes.flat[:9]):

    ax.contour(x, y, DEMup, levels=50, colors='gray', linewidths=0.5)
    ax.plot(xs, ys, 'ko', markersize=3)

    pcm = ax.pcolormesh(
        x,
        y,
        hsol[i*10],
        shading='auto',
        cmap='viridis',
        vmin=0,
        vmax=hmax,
        alpha=0.7
    )

    ax.set_title(f't = {t[i*10]/3600:.1f} hr')
    ax.set_aspect('equal')

# fig.colorbar(pcm, ax=axes.ravel().tolist(), shrink=0.8)
plt.tight_layout()
plt.show()